In [15]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

# ---------------------------------------------------------------------
# Input files
# ---------------------------------------------------------------------
# The notebook first looks in the current directory, then in /mnt/data.
# This makes it work both in Jupyter locally and in this ChatGPT sandbox.
CANDIDATE_DIRS = [Path("."), Path("/mnt/data")]

def find_file(filename):
    for d in CANDIDATE_DIRS:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Put it next to this notebook or in /mnt/data.")

TAXONOMY_TSV = find_file("../results/taxonomy_qiime/silva_138_99/8ab6ccd5-fad5-4d23-95c5-6de04f3f40dc/data/taxonomy.tsv")
MC_ORDER7_FILE = find_file("../results/speciate_it_classification/MC_order7_results.txt")

OUTDIR = Path("../results/taxonomy_adjustment_outputs")
OUTDIR.mkdir(exist_ok=True)

# Species calls below this MC confidence are downgraded to genus before comparison.
MIN_MC_CONFIDENCE_FOR_SPECIES = 0.70

print("QIIME taxonomy:", TAXONOMY_TSV)
print("MC order7 file:", MC_ORDER7_FILE)
print("Output folder:", OUTDIR.resolve())
print("Minimum MC confidence for species-level use:", MIN_MC_CONFIDENCE_FOR_SPECIES)

QIIME taxonomy: ../results/taxonomy_qiime/silva_138_99/8ab6ccd5-fad5-4d23-95c5-6de04f3f40dc/data/taxonomy.tsv
MC order7 file: ../results/speciate_it_classification/MC_order7_results.txt
Output folder: /home/rare/arlen/16S_colombian_vaginal_microbiome/results/taxonomy_adjustment_outputs
Minimum MC confidence for species-level use: 0.7


## 1. Load the two taxonomy files

`taxonomy.tsv` should have QIIME columns:

- `Feature ID`
- `Taxon`
- `Confidence`

`MC_order7_results.txt` is assumed to have four tab-separated columns without a header:

- `Feature ID`
- `MC_taxon`
- `MC_confidence`
- `MC_order7_group`

In [16]:
taxonomy = pd.read_csv(TAXONOMY_TSV, sep="\t")

mc = pd.read_csv(
    MC_ORDER7_FILE,
    sep="\t",
    header=None,
    names=["Feature ID", "MC_taxon", "MC_confidence", "MC_order7_group"]
)

print("taxonomy.tsv shape:", taxonomy.shape)
print("MC_order7 shape:", mc.shape)

display(taxonomy.head())
display(mc.head())

taxonomy.tsv shape: (939, 3)
MC_order7 shape: (939, 4)


,Feature ID,Taxon,Confidence
0,f6c0d5e255fc23b608315df8e9efe557,d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Gardnerella;s__,0.999992
1,64b421b8072426aaddfb8b937fa0ed27,d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Gardnerella;s__,0.999989
2,b24cd5a63c535ff4e55a2506d3cd62bb,d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Gardnerella;s__,0.999998
3,be014e630e00e5c6e3f0ca87daa86d1b,d__Bacteria;p__Actinobacteriota;c__Coriobacteriia;o__Coriobacteriales;f__Atopobiaceae;g__Atopobium;s__,0.994023
4,96f9d59d4d210fbeb71ba39924965ac1,d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Gardnerella;s__,0.999997


,Feature ID,MC_taxon,MC_confidence,MC_order7_group
0,f6c0d5e255fc23b608315df8e9efe557,Gardnerella_vaginalis,0.97,27
1,64b421b8072426aaddfb8b937fa0ed27,Gardnerella_vaginalis,0.97,27
2,b24cd5a63c535ff4e55a2506d3cd62bb,Gardnerella_vaginalis,0.98,27
3,be014e630e00e5c6e3f0ca87daa86d1b,Fannyhessea_vaginae,0.98,24
4,96f9d59d4d210fbeb71ba39924965ac1,Gardnerella_vaginalis,0.98,27


## 2. Basic validation

This step checks duplicated feature IDs and confirms whether both files contain the same features.

In [17]:
required_tax_cols = {"Feature ID", "Taxon", "Confidence"}
missing = required_tax_cols - set(taxonomy.columns)
if missing:
    raise ValueError(f"taxonomy.tsv is missing required columns: {missing}")

if taxonomy["Feature ID"].duplicated().any():
    dup_n = taxonomy["Feature ID"].duplicated().sum()
    raise ValueError(f"taxonomy.tsv contains {dup_n} duplicated Feature IDs.")

if mc["Feature ID"].duplicated().any():
    dup_n = mc["Feature ID"].duplicated().sum()
    raise ValueError(f"MC_order7_results.txt contains {dup_n} duplicated Feature IDs.")

taxonomy_ids = set(taxonomy["Feature ID"])
mc_ids = set(mc["Feature ID"])

print("Features in taxonomy.tsv:", len(taxonomy_ids))
print("Features in MC_order7:", len(mc_ids))
print("Overlapping features:", len(taxonomy_ids & mc_ids))
print("Only in taxonomy.tsv:", len(taxonomy_ids - mc_ids))
print("Only in MC_order7:", len(mc_ids - taxonomy_ids))

Features in taxonomy.tsv: 939
Features in MC_order7: 939
Overlapping features: 939
Only in taxonomy.tsv: 0
Only in MC_order7: 0


## 3. Helper functions

These functions parse the deepest taxonomic rank from QIIME and MC order-7.

Important details:

- QIIME taxonomy uses prefixes such as `d__`, `p__`, `g__`, `s__`.
- MC order-7 may report:
  - species-like labels such as `Gardnerella_vaginalis`, `Lactobacillus_iners`, `Prevotella_sp000479005`
  - genus-only labels such as `g_Prevotella`
  - higher-rank labels such as `f_Actinomycetaceae`, `o_Bacteroidales`, `d_Bacteria`
- Species-level MC labels below `MIN_MC_CONFIDENCE_FOR_SPECIES` are downgraded to genus before comparison.

In [18]:
RANK_ORDER = {
    "unassigned": 0,
    "domain": 1,
    "phylum": 2,
    "class": 3,
    "order": 4,
    "family": 5,
    "genus": 6,
    "species": 7,
}

QIIME_PREFIX_RANK = {
    "d__": "domain",
    "p__": "phylum",
    "c__": "class",
    "o__": "order",
    "f__": "family",
    "g__": "genus",
    "s__": "species",
}

MC_PREFIX_RANK = {
    "d_": "domain",
    "p_": "phylum",
    "c_": "class",
    "o_": "order",
    "f_": "family",
    "g_": "genus",
    "s_": "species",
}

RANK_PREFIXES = [
    ("d", "domain"),
    ("p", "phylum"),
    ("c", "class"),
    ("o", "order"),
    ("f", "family"),
    ("g", "genus"),
    ("s", "species"),
]

def parse_qiime_taxonomy(taxon):
    """Return parsed QIIME ranks, deepest rank, rank score, and deepest taxon name."""
    ranks = {rank: "" for _, rank in RANK_PREFIXES}

    if pd.isna(taxon) or str(taxon).strip() == "" or str(taxon).strip().lower() == "unassigned":
        return ranks, "unassigned", 0, ""

    parts = [p.strip() for p in str(taxon).split(";")]
    max_rank = "unassigned"
    max_taxon = ""

    for part in parts:
        if not part:
            continue

        for prefix, rank in QIIME_PREFIX_RANK.items():
            if part.startswith(prefix):
                value = part[len(prefix):].strip()
                ranks[rank] = value

                if value and value.lower() not in {"unassigned", "uncultured", "metagenome"}:
                    max_rank = rank
                    max_taxon = value
                break

    return ranks, max_rank, RANK_ORDER[max_rank], max_taxon


def infer_mc_rank(mc_taxon):
    """Infer taxonomic rank from an MC order-7 label."""
    if pd.isna(mc_taxon) or str(mc_taxon).strip() == "":
        return "unassigned", 0, ""

    taxon = str(mc_taxon).strip()

    # Explicit rank prefixes: g_Prevotella, f_Actinomycetaceae, d_Bacteria, etc.
    for prefix, rank in MC_PREFIX_RANK.items():
        if taxon.startswith(prefix):
            value = taxon[len(prefix):].strip()
            return rank, RANK_ORDER[rank], value

    # No prefix: interpret underscore-separated binomial or GTDB-style placeholder species
    # as species level, e.g. Gardnerella_vaginalis, Prevotella_sp000479005,
    # Ca_Lachnocurva_vaginae.
    parts = [p for p in taxon.split("_") if p]
    if len(parts) >= 2:
        return "species", RANK_ORDER["species"], taxon

    # Conservative fallback.
    return "genus", RANK_ORDER["genus"], taxon


def mc_genus_from_taxon(mc_taxon, mc_rank=None):
    """Infer genus from an MC taxon, including species-level labels."""
    if pd.isna(mc_taxon):
        return ""

    taxon = str(mc_taxon).strip()

    if mc_rank is None:
        mc_rank = infer_mc_rank(taxon)[0]

    if mc_rank == "genus":
        return re.sub(r"^g_", "", taxon)

    if mc_rank == "species":
        parts = [p for p in taxon.split("_") if p]

        # Candidatus abbreviation, e.g. Ca_Lachnocurva_vaginae
        if len(parts) >= 3 and parts[0] in {"Ca", "Candidatus"}:
            return parts[1]

        if parts:
            return parts[0]

    return ""


def normalize_name(x):
    if pd.isna(x):
        return ""

    s = str(x).strip()
    s = re.sub(r"^[a-z]__", "", s)
    s = re.sub(r"^[a-z]_", "", s)
    return s.lower()


def qiime_standard_taxon_from_ranks(ranks):
    """Create a seven-rank QIIME-style taxonomy string."""
    return ";".join([f"{prefix}__{ranks.get(rank, '') or ''}" for prefix, rank in RANK_PREFIXES])


def standardize_qiime_taxon(taxon):
    ranks, _, _, _ = parse_qiime_taxonomy(taxon)
    return qiime_standard_taxon_from_ranks(ranks)


def accepted_mc_assignment(mc_taxon, mc_confidence, min_species_conf=0.70):
    """Apply the species confidence rule to MC order-7 assignments."""
    raw_rank, raw_score, raw_value = infer_mc_rank(mc_taxon)
    raw_taxon = "" if pd.isna(mc_taxon) else str(mc_taxon).strip()

    accepted_rank = raw_rank
    accepted_taxon = raw_taxon
    downgraded = False
    downgrade_note = ""

    if raw_rank == "species" and pd.notna(mc_confidence):
        if float(mc_confidence) < min_species_conf:
            genus = mc_genus_from_taxon(raw_taxon, "species")
            if genus:
                accepted_rank = "genus"
                accepted_taxon = f"g_{genus}"
                downgraded = True
                downgrade_note = (
                    f"MC_order7 had species label but confidence {float(mc_confidence):.2f} "
                    f"< {min_species_conf}; downgraded to genus."
                )

    return {
        "raw_rank": raw_rank,
        "raw_score": raw_score,
        "raw_taxon": raw_taxon,
        "accepted_rank": accepted_rank,
        "accepted_score": RANK_ORDER[accepted_rank],
        "accepted_taxon": accepted_taxon,
        "downgraded": downgraded,
        "downgrade_note": downgrade_note,
    }


def build_qiime_with_mc(qiime_taxon, mc_taxon, mc_rank):
    """Insert the accepted MC assignment into the QIIME lineage."""
    ranks, _, _, _ = parse_qiime_taxonomy(qiime_taxon)

    if mc_rank == "species":
        mc_genus = mc_genus_from_taxon(mc_taxon, "species")
        if mc_genus:
            ranks["genus"] = mc_genus
        ranks["species"] = str(mc_taxon).strip()

    elif mc_rank == "genus":
        genus = re.sub(r"^g_", "", str(mc_taxon).strip())
        ranks["genus"] = genus
        ranks["species"] = ""

    elif mc_rank in ["family", "order", "class", "phylum", "domain"]:
        value = re.sub(r"^[dpcofgs]_", "", str(mc_taxon).strip())
        ranks[mc_rank] = value

        rank_list = [rank for _, rank in RANK_PREFIXES]
        idx = rank_list.index(mc_rank)

        # If this higher-rank MC assignment becomes the final choice,
        # clear downstream ranks to avoid mixing unsupported lower ranks.
        for downstream_rank in rank_list[idx + 1:]:
            ranks[downstream_rank] = ""

    return qiime_standard_taxon_from_ranks(ranks)

## 4. Merge QIIME and MC order-7 by Feature ID

In [19]:
merged = taxonomy.merge(
    mc,
    on="Feature ID",
    how="outer",
    validate="one_to_one"
)

print("Merged table shape:", merged.shape)
display(merged.head())

Merged table shape: (939, 6)


,Feature ID,Taxon,Confidence,MC_taxon,MC_confidence,MC_order7_group
0,006f75621a34a4ce85537c3693dc11d6,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__,0.784519,g_Prevotella,0.66,54
1,009160bfde60715772e504df6b7b4d40,Unassigned,0.394273,d_Bacteria,0.24,13
2,01316bf6c7369ec6e12e5244d142e220,d__Bacteria,0.819933,d_Bacteria,0.24,13
3,013ecd818bc4a67b41808376d4d7b8e0,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae,0.999364,Prevotella_bivia,0.97,54
4,01ca864e183da7caf55fd0cfdbb7f1a8,d__Bacteria;p__Firmicutes;c__Clostridia;o__Peptostreptococcales-Tissierellales,0.825896,g_Anaerococcus,0.52,74


## 5. Choose the maximum taxonomic depth per feature

Tie policy:

- If both reach species: use MC order-7 as the species refinement.
- If both reach genus or a higher rank: keep QIIME lineage because it preserves the full upstream taxonomy.
- If one file reaches a deeper rank than the other: keep the deeper one.

In [20]:
def choose_best_taxonomy(row, min_mc_species_conf=0.70):
    q_ranks, q_rank, q_score, q_max_taxon = parse_qiime_taxonomy(row["Taxon"])

    mc_info = accepted_mc_assignment(
        row["MC_taxon"],
        row["MC_confidence"],
        min_species_conf=min_mc_species_conf
    )

    raw_mc_rank = mc_info["raw_rank"]
    raw_mc_score = mc_info["raw_score"]
    raw_mc_taxon = mc_info["raw_taxon"]

    mc_rank = mc_info["accepted_rank"]
    mc_score = mc_info["accepted_score"]
    mc_taxon_accepted = mc_info["accepted_taxon"]

    q_genus = q_ranks.get("genus", "")
    mc_genus = mc_genus_from_taxon(raw_mc_taxon, raw_mc_rank)

    if q_genus and mc_genus:
        genus_match = normalize_name(q_genus) == normalize_name(mc_genus)
    else:
        genus_match = np.nan

    # Choose deepest assignment.
    if mc_score > q_score:
        final_source = "MC_order7"
        final_rank = mc_rank
        final_rank_score = mc_score
        final_taxon = build_qiime_with_mc(row["Taxon"], mc_taxon_accepted, mc_rank)
        final_confidence = row["MC_confidence"]
        decision_note = f"MC_order7 is deeper ({mc_rank}) than QIIME ({q_rank})."

    elif q_score > mc_score:
        final_source = "QIIME"
        final_rank = q_rank
        final_rank_score = q_score
        final_taxon = standardize_qiime_taxon(row["Taxon"])
        final_confidence = row["Confidence"]
        decision_note = f"QIIME is deeper ({q_rank}) than MC_order7 ({mc_rank})."

    else:
        # Same depth.
        if mc_rank == "species":
            final_source = "MC_order7"
            final_rank = mc_rank
            final_rank_score = mc_score
            final_taxon = build_qiime_with_mc(row["Taxon"], mc_taxon_accepted, mc_rank)
            final_confidence = row["MC_confidence"]
            decision_note = "Same depth at species; MC_order7 used as species-level refinement."
        else:
            final_source = "QIIME"
            final_rank = q_rank
            final_rank_score = q_score
            final_taxon = standardize_qiime_taxon(row["Taxon"])
            final_confidence = row["Confidence"]
            decision_note = f"Same depth ({q_rank}); QIIME lineage retained."

    if mc_info["downgrade_note"]:
        decision_note = mc_info["downgrade_note"] + " " + decision_note

    return pd.Series({
        "QIIME_rank": q_rank,
        "QIIME_rank_score": q_score,
        "QIIME_max_taxon": q_max_taxon,
        "QIIME_genus": q_genus,

        "MC_raw_rank": raw_mc_rank,
        "MC_raw_rank_score": raw_mc_score,
        "MC_rank": mc_rank,
        "MC_rank_score": mc_score,
        "MC_taxon_accepted": mc_taxon_accepted,
        "MC_genus_inferred": mc_genus,
        "MC_species_downgraded_by_confidence": mc_info["downgraded"],
        "Genus_match_if_comparable": genus_match,

        "Final_Taxon": final_taxon,
        "Final_Confidence": final_confidence,
        "Final_Rank": final_rank,
        "Final_Rank_Score": final_rank_score,
        "Final_Source": final_source,
        "Decision_Note": decision_note,
    })


decision_table = merged.apply(
    choose_best_taxonomy,
    axis=1,
    min_mc_species_conf=MIN_MC_CONFIDENCE_FOR_SPECIES
)

combined = pd.concat([merged, decision_table], axis=1)

display(combined.head())

,Feature ID,Taxon,Confidence,MC_taxon,MC_confidence,MC_order7_group,QIIME_rank,QIIME_rank_score,QIIME_max_taxon,QIIME_genus,MC_raw_rank,MC_raw_rank_score,MC_rank,MC_rank_score,MC_taxon_accepted,MC_genus_inferred,MC_species_downgraded_by_confidence,Genus_match_if_comparable,Final_Taxon,Final_Confidence,Final_Rank,Final_Rank_Score,Final_Source,Decision_Note
0,006f75621a34a4ce85537c3693dc11d6,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__,0.784519,g_Prevotella,0.66,54,genus,6,Prevotella,Prevotella,genus,6,genus,6,g_Prevotella,Prevotella,False,True,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__,0.784519,genus,6,QIIME,Same depth (genus); QIIME lineage retained.
1,009160bfde60715772e504df6b7b4d40,Unassigned,0.394273,d_Bacteria,0.24,13,unassigned,0,,,domain,1,domain,1,d_Bacteria,,False,NaN,d__Bacteria;p__;c__;o__;f__;g__;s__,0.240000,domain,1,MC_order7,MC_order7 is deeper (domain) than QIIME (unassigned).
2,01316bf6c7369ec6e12e5244d142e220,d__Bacteria,0.819933,d_Bacteria,0.24,13,domain,1,Bacteria,,domain,1,domain,1,d_Bacteria,,False,NaN,d__Bacteria;p__;c__;o__;f__;g__;s__,0.819933,domain,1,QIIME,Same depth (domain); QIIME lineage retained.
3,013ecd818bc4a67b41808376d4d7b8e0,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae,0.999364,Prevotella_bivia,0.97,54,family,5,Prevotellaceae,,species,7,species,7,Prevotella_bivia,Prevotella,False,NaN,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_bivia,0.970000,species,7,MC_order7,MC_order7 is deeper (species) than QIIME (family).
4,01ca864e183da7caf55fd0cfdbb7f1a8,d__Bacteria;p__Firmicutes;c__Clostridia;o__Peptostreptococcales-Tissierellales,0.825896,g_Anaerococcus,0.52,74,order,4,Peptostreptococcales-Tissierellales,,genus,6,genus,6,g_Anaerococcus,Anaerococcus,False,NaN,d__Bacteria;p__Firmicutes;c__Clostridia;o__Peptostreptococcales-Tissierellales;f__;g__Anaerococcus;s__,0.520000,genus,6,MC_order7,MC_order7 is deeper (genus) than QIIME (order).


## 6. Check whether MC order-7 reaches species level

This summary separates:

- `MC_raw_rank`: what the MC file originally reported.
- `MC_rank`: accepted rank after applying the species confidence threshold.
- `Final_Rank`: rank used in the final adjusted taxonomy.

In [21]:
print("MC raw rank counts:")
display(combined["MC_raw_rank"].value_counts().rename_axis("rank").reset_index(name="n"))

print("MC accepted rank counts after confidence filter:")
display(combined["MC_rank"].value_counts().rename_axis("rank").reset_index(name="n"))

print("QIIME maximum rank counts:")
display(combined["QIIME_rank"].value_counts().rename_axis("rank").reset_index(name="n"))

print("Final rank counts:")
display(combined["Final_Rank"].value_counts().rename_axis("rank").reset_index(name="n"))

print("Final source counts:")
display(combined["Final_Source"].value_counts().rename_axis("source").reset_index(name="n"))

raw_species_n = int((combined["MC_raw_rank"] == "species").sum())
accepted_species_n = int((combined["MC_rank"] == "species").sum())
final_species_n = int((combined["Final_Rank"] == "species").sum())
total_n = len(combined)

print(f"MC_order7 raw species-level calls: {raw_species_n}/{total_n} ({raw_species_n / total_n:.1%})")
print(f"MC_order7 accepted species-level calls after threshold: {accepted_species_n}/{total_n} ({accepted_species_n / total_n:.1%})")
print(f"Final species-level taxonomy rows: {final_species_n}/{total_n} ({final_species_n / total_n:.1%})")
print(f"MC species calls downgraded by confidence threshold: {int(combined['MC_species_downgraded_by_confidence'].sum())}")

MC raw rank counts:


,rank,n
0,species,391
1,domain,246
2,genus,204
3,family,64
4,class,17
5,order,17


MC accepted rank counts after confidence filter:


,rank,n
0,species,387
1,domain,246
2,genus,208
3,family,64
4,class,17
5,order,17


QIIME maximum rank counts:


...

Final rank counts:


,rank,n
0,genus,392
1,species,387
2,domain,138
3,family,15
4,class,6
5,order,1


Final source counts:


,source,n
0,MC_order7,572
1,QIIME,367


MC_order7 raw species-level calls: 391/939 (41.6%)
MC_order7 accepted species-level calls after threshold: 387/939 (41.2%)
Final species-level taxonomy rows: 387/939 (41.2%)
MC species calls downgraded by confidence threshold: 4


## 7. Inspect species-level MC assignments

These are the rows where MC order-7 reached species level before the confidence filter.

In [22]:
mc_species = combined[combined["MC_raw_rank"] == "species"].copy()

display(
    mc_species[
        [
            "Feature ID",
            "MC_taxon",
            "MC_confidence",
            "MC_species_downgraded_by_confidence",
            "QIIME_rank",
            "Final_Rank",
            "Final_Source",
            "Final_Taxon",
        ]
    ].head(20)
)

,Feature ID,MC_taxon,MC_confidence,MC_species_downgraded_by_confidence,QIIME_rank,Final_Rank,Final_Source,Final_Taxon
3,013ecd818bc4a67b41808376d4d7b8e0,Prevotella_bivia,0.97,False,family,species,MC_order7,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_bivia
5,024d64ec6ca834ca7a0889e9b43cce4b,Bifidobacterium_dentium,0.97,False,genus,species,MC_order7,d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Bifidobacterium;s__Bifidobacterium_dentium
17,0792b6d3dee4eb623d00869e9dbeab7d,Prevotella_colorans,0.91,False,genus,species,MC_order7,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_colorans
19,0864d017c1c1491d59694f109bc5a68f,Sneathia_vaginalis,0.92,False,genus,species,MC_order7,d__Bacteria;p__Fusobacteriota;c__Fusobacteriia;o__Fusobacteriales;f__Leptotrichiaceae;g__Sneathia;s__Sneathia_vaginalis
21,08a722ea32f62a771b6aa38df0ef0254,Amygdalobacter_nucleatus,0.85,False,genus,species,MC_order7,d__Bacteria;p__Firmicutes;c__Clostridia;o__Oscillospirales;f__Hungateiclostridiaceae;g__Amygdalobacter;s__Amygdalobacter_nucleatus
22,08f097cb51a3d06b4f547925e3d2919a,Peptoniphilus_duerdenii,0.95,False,genus,species,MC_order7,d__Bacteria;p__Firmicutes;c__Clostridia;o__Peptostreptococcales-Tissierellales;f__Family_XI;g__Peptoniphilus;s__Peptoniphilus_duerdenii
23,096e693716ee1b2035941b35c5a73150,Corynebacterium_sp001767255,0.95,False,genus,species,MC_order7,d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Corynebacteriales;f__Corynebacteriaceae;g__Corynebacterium;s__Corynebacterium_sp001...
25,0a05309d0f3f388b992277573b5314a8,Prevotella_bivia,0.93,False,family,species,MC_order7,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_bivia
27,0ae1308cf1201de65c4edfcd616e89b2,Eisenbergiella_sp900550285,0.69,True,genus,genus,QIIME,d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__Fusicatenibacter;s__
29,0b311fd10d1a4071f1c20a67adaa55ac,Prevotella_copri,0.97,False,genus,species,MC_order7,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_copri


## 8. Inspect MC-vs-QIIME genus mismatches

These are important for vaginal microbiome data because QIIME/SILVA and MC/GTDB-style assignments may use different genus names, for example `Atopobium` versus `Fannyhessea`.

When MC is accepted at species level, the notebook updates the genus in the final taxonomy string to match the MC species label.

In [23]:
genus_mismatches = combined[
    (combined["MC_raw_rank"] == "species") &
    (combined["QIIME_genus"].fillna("") != "") &
    (combined["MC_genus_inferred"].fillna("") != "") &
    (combined["Genus_match_if_comparable"] == False)
].copy()

print("MC species-level rows with QIIME-vs-MC genus mismatch:", len(genus_mismatches))

display(
    genus_mismatches[
        [
            "Feature ID",
            "QIIME_genus",
            "MC_taxon",
            "MC_genus_inferred",
            "Confidence",
            "MC_confidence",
            "Final_Taxon",
        ]
    ].head(30)
)

MC species-level rows with QIIME-vs-MC genus mismatch: 97


,Feature ID,QIIME_genus,MC_taxon,MC_genus_inferred,Confidence,MC_confidence,Final_Taxon
17,0792b6d3dee4eb623d00869e9dbeab7d,Prevotella_7,Prevotella_colorans,Prevotella,0.900892,0.91,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_colorans
21,08a722ea32f62a771b6aa38df0ef0254,Fastidiosipila,Amygdalobacter_nucleatus,Amygdalobacter,0.771159,0.85,d__Bacteria;p__Firmicutes;c__Clostridia;o__Oscillospirales;f__Hungateiclostridiaceae;g__Amygdalobacter;s__Amygdalobacter_nucleatus
27,0ae1308cf1201de65c4edfcd616e89b2,Fusicatenibacter,Eisenbergiella_sp900550285,Eisenbergiella,0.996309,0.69,d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__Fusicatenibacter;s__
29,0b311fd10d1a4071f1c20a67adaa55ac,Prevotella_9,Prevotella_copri,Prevotella,0.999988,0.97,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_copri
37,0ca6a9c5b1b03d6c643666062c82b499,[Eubacterium]_coprostanoligenes_group,Blautia_A_sp018919065,Blautia,0.999500,0.92,d__Bacteria;p__Firmicutes;c__Clostridia;o__Oscillospirales;f__[Eubacterium]_coprostanoligenes_group;g__Blautia;s__Blautia_A_sp018919065
41,0ddd7b91e7368df6c751335a155f4cb2,Atopobium,Fannyhessea_vaginae,Fannyhessea,0.798756,0.81,d__Bacteria;p__Actinobacteriota;c__Coriobacteriia;o__Coriobacteriales;f__Atopobiaceae;g__Fannyhessea;s__Fannyhessea_vaginae
49,0f9295744170f03f0446812dee684099,[Eubacterium]_eligens_group,Lachnospira_eligens_A,Lachnospira,0.988849,0.97,d__Bacteria;p__Firmicutes;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__Lachnospira;s__Lachnospira_eligens_A
75,154741a52d5412be964f4499c769204e,Prevotella_9,Prevotella_copri,Prevotella,0.998786,0.70,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_copri
85,18fed6e2ac6209143110497142b9635d,Mycoplasma,Metamycoplasma_hominis,Metamycoplasma,0.999985,0.92,d__Bacteria;p__Firmicutes;c__Bacilli;o__Mycoplasmatales;f__Mycoplasmataceae;g__Metamycoplasma;s__Metamycoplasma_hominis
90,196a7fa4e2ec386be9c8d33942c75407,Saccharimonadales,Nanoperiomorbus_sp004136275,Nanoperiomorbus,0.999901,0.97,d__Bacteria;p__Patescibacteria;c__Saccharimonadia;o__Saccharimonadales;f__Saccharimonadales;g__Nanoperiomorbus;s__Nanoperiomorbus_sp0041...


## 9. Create final adjusted QIIME-compatible taxonomy table

The final table has the same core structure as QIIME's exported `taxonomy.tsv`:

- `Feature ID`
- `Taxon`
- `Confidence`

`Confidence` comes from the source chosen for the final assignment:

- MC confidence when MC order-7 is selected.
- QIIME confidence when QIIME is selected.

In [24]:
adjusted_taxonomy = combined[
    ["Feature ID", "Final_Taxon", "Final_Confidence"]
].rename(columns={
    "Final_Taxon": "Taxon",
    "Final_Confidence": "Confidence",
})

display(adjusted_taxonomy.head(20))

adjusted_taxonomy.to_csv(OUTDIR / "adjusted_taxonomy.tsv", sep="\t", index=False)
combined.to_csv(OUTDIR / "taxonomy_adjustment_detailed.tsv", sep="\t", index=False)

print("Saved:", OUTDIR / "adjusted_taxonomy.tsv")
print("Saved:", OUTDIR / "taxonomy_adjustment_detailed.tsv")

,Feature ID,Taxon,Confidence
0,006f75621a34a4ce85537c3693dc11d6,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__,0.784519
1,009160bfde60715772e504df6b7b4d40,d__Bacteria;p__;c__;o__;f__;g__;s__,0.240000
2,01316bf6c7369ec6e12e5244d142e220,d__Bacteria;p__;c__;o__;f__;g__;s__,0.819933
3,013ecd818bc4a67b41808376d4d7b8e0,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__Prevotella_bivia,0.970000
4,01ca864e183da7caf55fd0cfdbb7f1a8,d__Bacteria;p__Firmicutes;c__Clostridia;o__Peptostreptococcales-Tissierellales;f__;g__Anaerococcus;s__,0.520000
5,024d64ec6ca834ca7a0889e9b43cce4b,d__Bacteria;p__Actinobacteriota;c__Actinobacteria;o__Bifidobacteriales;f__Bifidobacteriaceae;g__Bifidobacterium;s__Bifidobacterium_dentium,0.970000
6,02e0e7cf8aa8041880c82e640ed3c14e,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Marinifilaceae;g__Odoribacter;s__,1.000000
7,031c22fdc2cc0af9d00d62419ada611a,d__Bacteria;p__Firmicutes;c__Negativicutes;o__Veillonellales-Selenomonadales;f__Veillonellaceae;g__Megasphaera;s__,0.725771
8,03599b87a1380fe9624bacf6502041fe,d__Bacteria;p__Firmicutes;c__Negativicutes;o__Veillonellales-Selenomonadales;f__Veillonellaceae;g__Dialister;s__,0.924370
9,036ca3fa03f8d93a402ced6240a337fe,d__Bacteria;p__Firmicutes;c__Clostridia;o__Oscillospirales;f__Oscillospiraceae;g__UCG-002;s__,0.993556


Saved: ../results/taxonomy_adjustment_outputs/adjusted_taxonomy.tsv
Saved: ../results/taxonomy_adjustment_outputs/taxonomy_adjustment_detailed.tsv


## 10. Export QC tables

In [25]:
# MC species-level rows, including those downgraded by confidence threshold.
combined[combined["MC_raw_rank"] == "species"][
    [
        "Feature ID",
        "MC_taxon",
        "MC_confidence",
        "MC_order7_group",
        "MC_species_downgraded_by_confidence",
        "MC_taxon_accepted",
        "Taxon",
        "Confidence",
        "Final_Taxon",
        "Final_Rank",
        "Final_Source",
    ]
].to_csv(OUTDIR / "mc_order7_species_level_assignments.tsv", sep="\t", index=False)

# Rows that remain unresolved below genus.
combined[combined["Final_Rank"].isin(["unassigned", "domain", "phylum", "class", "order", "family"])][
    [
        "Feature ID",
        "Taxon",
        "Confidence",
        "MC_taxon",
        "MC_confidence",
        "QIIME_rank",
        "MC_rank",
        "Final_Taxon",
        "Final_Rank",
        "Final_Source",
        "Decision_Note",
    ]
].to_csv(OUTDIR / "features_not_resolved_to_genus_or_species.tsv", sep="\t", index=False)

# QIIME-vs-MC genus mismatches among MC species-level rows.
genus_mismatches[
    [
        "Feature ID",
        "QIIME_genus",
        "MC_taxon",
        "MC_genus_inferred",
        "Taxon",
        "Confidence",
        "MC_confidence",
        "Final_Taxon",
    ]
].to_csv(OUTDIR / "qiime_mc_genus_mismatches_when_mc_species.tsv", sep="\t", index=False)

# Summary tables.
combined["MC_raw_rank"].value_counts().rename_axis("rank").reset_index(name="n").to_csv(
    OUTDIR / "mc_raw_rank_counts.tsv", sep="\t", index=False
)
combined["MC_rank"].value_counts().rename_axis("rank").reset_index(name="n").to_csv(
    OUTDIR / "mc_accepted_rank_counts.tsv", sep="\t", index=False
)
combined["QIIME_rank"].value_counts().rename_axis("rank").reset_index(name="n").to_csv(
    OUTDIR / "qiime_rank_counts.tsv", sep="\t", index=False
)
combined["Final_Rank"].value_counts().rename_axis("rank").reset_index(name="n").to_csv(
    OUTDIR / "final_rank_counts.tsv", sep="\t", index=False
)
combined["Final_Source"].value_counts().rename_axis("source").reset_index(name="n").to_csv(
    OUTDIR / "final_source_counts.tsv", sep="\t", index=False
)

print("QC files written to:", OUTDIR.resolve())
for path in sorted(OUTDIR.glob("*.tsv")):
    print("-", path.name)

QC files written to: /home/rare/arlen/16S_colombian_vaginal_microbiome/results/taxonomy_adjustment_outputs
- adjusted_taxonomy.tsv
- features_not_resolved_to_genus_or_species.tsv
- final_rank_counts.tsv
- final_source_counts.tsv
- mc_accepted_rank_counts.tsv
- mc_order7_species_level_assignments.tsv
- mc_raw_rank_counts.tsv
- qiime_mc_genus_mismatches_when_mc_species.tsv
- qiime_rank_counts.tsv
- taxonomy_adjustment_detailed.tsv


## 11. Final checks

This verifies that the adjusted taxonomy has the same number of rows and Feature IDs as the original taxonomy.

In [26]:
assert len(adjusted_taxonomy) == len(taxonomy), "Adjusted taxonomy row count differs from original taxonomy."
assert set(adjusted_taxonomy["Feature ID"]) == set(taxonomy["Feature ID"]), "Adjusted taxonomy Feature IDs differ from original taxonomy."

print("Final adjusted taxonomy is consistent with the original feature list.")
print("Rows:", len(adjusted_taxonomy))
print("Output:", OUTDIR / "adjusted_taxonomy.tsv")

Final adjusted taxonomy is consistent with the original feature list.
Rows: 939
Output: ../results/taxonomy_adjustment_outputs/adjusted_taxonomy.tsv


## 12. Optional: inspect one feature manually

Change `FEATURE_ID_TO_CHECK` to inspect how a specific ASV/feature was decided.

In [27]:
FEATURE_ID_TO_CHECK = combined.iloc[0]["Feature ID"]

combined.loc[
    combined["Feature ID"] == FEATURE_ID_TO_CHECK,
    [
        "Feature ID",
        "Taxon",
        "Confidence",
        "QIIME_rank",
        "MC_taxon",
        "MC_confidence",
        "MC_raw_rank",
        "MC_rank",
        "Final_Taxon",
        "Final_Confidence",
        "Final_Rank",
        "Final_Source",
        "Decision_Note",
    ]
]

,Feature ID,Taxon,Confidence,QIIME_rank,MC_taxon,MC_confidence,MC_raw_rank,MC_rank,Final_Taxon,Final_Confidence,Final_Rank,Final_Source,Decision_Note
0,006f75621a34a4ce85537c3693dc11d6,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__,0.784519,genus,g_Prevotella,0.66,genus,genus,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Prevotellaceae;g__Prevotella;s__,0.784519,genus,QIIME,Same depth (genus); QIIME lineage retained.
